In [1]:
# Initialize & Load Feature Engineered Data
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/heart_disease_features.csv')

X = df.drop(columns=['HeartDiseaseorAttack'])
y = df['HeartDiseaseorAttack']

In [2]:
train_X, val_X, train_y, val_y = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

train_X = train_X.drop(columns=['lifestyle_risk_score'])
val_X = val_X.drop(columns=['lifestyle_risk_score'])

train_y.value_counts(normalize=True)
val_y.value_counts(normalize=True)

HeartDiseaseorAttack
0.0    0.896795
1.0    0.103205
Name: proportion, dtype: float64

In [3]:
# Fit Baseline Models with Class-Imbalance handling

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=1, class_weight='balanced'),
    "LightGBM": LGBMClassifier(random_state=1, class_weight='balanced')
}

# class_weight = 'balanced' automatically adjusts weights inversely proportional to class frequencies in the input data.

for name, model in models.items():
    model.fit(train_X, train_y)

[LightGBM] [Info] Number of positive: 18974, number of negative: 164850
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 211
[LightGBM] [Info] Number of data points in the train set: 183824, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [4]:
# Sanity Check

for name, model in models.items():
    preds = model.predict(val_X)
    probs = model.predict_proba(val_X)[:, 1]
    print(name, preds[:10], probs[:10])

    # Confirms if every model fits and produces both a class prediction and a proba output

Logistic Regression [1. 1. 0. 1. 0. 0. 0. 0. 0. 1.] [0.70336075 0.70952524 0.09129349 0.91826057 0.16013625 0.2414385
 0.10166533 0.01905105 0.48392319 0.52972289]
Random Forest [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.] [0.37 0.31 0.   0.55 0.   0.03 0.01 0.   0.21 0.19]
LightGBM [1. 1. 0. 1. 0. 0. 0. 0. 0. 1.] [0.69696298 0.66767141 0.07953599 0.86489108 0.11690732 0.21947459
 0.0863663  0.01861853 0.46711079 0.60934438]


In [5]:
# Save models & Split (For notebook 04_model_validation)

import joblib 

joblib.dump(models, '../data/processed/baseline_models.pkl')
joblib.dump((train_X, val_X, train_y, val_y), '../data/processed/train_val_split.pkl')

['../data/processed/train_val_split.pkl']